# COMMONWEAVE Zero — Notebook de verificación

Este notebook comprueba de extremo a extremo:

1. Compilación del componente nativo C
2. Generación del adaptador desde el contrato YAML
3. Verificación diferencial (equivalencia semántica)
4. Benchmark de cuatro vías
5. Emisión y validación del sello de integridad (CSG / Ed25519 + SHA3-256)

**Requisitos:** Python ≥ 3.10, `gcc`, `make`, `pyyaml`, `cryptography`.

```bash
pip install pyyaml cryptography
```

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

# Ajusta si abres el notebook desde otro sitio
ROOT = Path("..").resolve()
if not (ROOT / "run_zero.py").exists():
    ROOT = Path(".").resolve()
assert (ROOT / "run_zero.py").exists(), f"No se encuentra run_zero.py en {ROOT}"
print("ROOT =", ROOT)
sys.path.insert(0, str(ROOT))

## 1. Ejecutar el pipeline completo

Equivale a `python run_zero.py`. Puede tardar ~10–30 s.

In [ ]:
result = subprocess.run(
    [sys.executable, "run_zero.py"],
    cwd=ROOT,
    capture_output=True,
    text=True,
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1500:])
print("\n>>> returncode =", result.returncode, "(0 = PASS)")
assert result.returncode == 0, "El pipeline falló"

## 2. Inspeccionar el certificado de aceptación

In [ ]:
acc = json.loads((ROOT / "report" / "acceptance.json").read_text())
print("Decisión:", acc["decision"])
print("Veredicto diferencial:", acc["differential"]["verdict"])
print("Pass rate:", acc["differential"]["pass_rate"])
print("Edge cases:", acc["differential"]["edge_passed"])
print()
print("Pruebas registradas:")
for p in acc["proofs"]:
    print(f"  [{p['status']}] {p['id']}: {p['statement'][:90]}…")
print()
print("Límites:", acc["limits"][:120], "…")

## 3. Verificar el sello de integridad (CSG)

In [ ]:
from seal.sello import SelloIntegridad

sello_path = ROOT / "report" / "sello.json"
sello_data = json.loads(sello_path.read_text())
vr = SelloIntegridad.verificar(sello_data, base_dir=ROOT)
print(json.dumps(vr, indent=2))
assert vr["signature_valid"] and vr["artifacts_intact"], "Sello inválido"
print("\n✓ Sello válido: firma Ed25519 + digests SHA3-256 de todos los artefactos")

## 4. Resumen del benchmark (latencia mediana)

In [ ]:
bench = json.loads((ROOT / "report" / "benchmark.json").read_text())
print(f"{'Size':>8}  {'pure_py':>10}  {'manual':>10}  {'generated':>10}  {'ipc':>10}")
print("-" * 54)
for size, block in bench["implementations"].items():
    row = [f"{size}x{size}"]
    for name in ["pure_python", "manual_ctypes", "generated", "ipc"]:
        med_us = block[name]["median"] * 1e6
        row.append(f"{med_us:8.1f} µs")
    print("  ".join(f"{c:>10}" for c in row))

print("\nEsfuerzo (proxy LOC de pegamento):")
for k, v in bench["effort_proxy"].items():
    print(f"  {k:16s}  loc_glue={v['loc_glue']:3d}  ({v['note']})")

## 5. Conclusión rápida

- **Viabilidad:** el adaptador se genera desde el contrato y pasa la verificación diferencial (60/60 + 5/5 edge).
- **Rendimiento:** el adaptador generado es competitivo con el binding ctypes manual y mucho más rápido que Python puro / IPC a partir de 8×8.
- **Esfuerzo:** el camino generado requiere 0 LOC de pegamento (solo el contrato); el manual ~25 LOC.
- **Evidencia:** el sello CSG permite a un tercero comprobar que los informes no han sido alterados.

Si todas las celdas anteriores terminan sin error, COMMONWEAVE Zero funciona correctamente en este entorno.